# GEARS

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU
from torch_geometric.nn import SGConv

## MLP
可以重复使用的多层感知机 MLP。提供每层的维度，能自动组装出一个小网络。

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, sizes, batch_norm=True, last_layer_act="linear"):
        super(MLP, self).__init__()
        # sizes是列表，存储每层的维度数
        layers = [] # 存储层
        for s in range(len(sizes) - 1):
            layers = layers + [
                torch.nn.Linear(sizes[s], sizes[s + 1]),
                torch.nn.BatchNorm1d(sizes[s + 1]) if batch_norm and s < len(sizes) - 1 else None,
                # s < len(sizes) - 1恒成立，只需batch_norm为True则会加入BatchNorm1d
                torch.nn.ReLU()
            ]

        layers = [l for l in layers if l is not None][:-1] # 遍历，保留所有不为None的层，并不取最后一个层（最后一个ReLU）
        self.activation = last_layer_act
        self.network = torch.nn.Sequential(*layers) # 解包再组装
        self.relu = torch.nn.ReLU() # 未使用
    def forward(self, x):
        return self.network(x)

## GEARS_Model

In [ ]:
class GEARS_Model(torch.nn.Module):

    def __init__(self, args):
        # args是模型的配置信息字典，参数的值从字典中读取
        super(GEARS_Model, self).__init__()
        self.args = args       
        self.num_genes = args['num_genes'] # 基因数，本例5054
        self.num_perts = args['num_perts'] # 扰动图基因节点数
        hidden_size = args['hidden_size'] # 隐层数，本例64
        self.uncertainty = args['uncertainty'] # 开关是否启用额外的不确定性预测分支
        self.num_layers = args['num_go_gnn_layers'] # GO扰动图的图卷积层数，本例为1
        self.indv_out_hidden_size = args['decoder_hidden_size'] # 解码器隐藏维度，本例16
        self.num_layers_gene_pos = args['num_gene_gnn_layers'] # 基因共表达图分支的图卷积层数，本例为1
        self.no_perturb = args['no_perturb'] # 是否走“不预测扰动变化”的特殊路径
        self.pert_emb_lambda = 0.2
        
        self.pert_w = nn.Linear(1, hidden_size) # 扰动的值转为64维与嵌入向量匹配
                       
        self.gene_emb = nn.Embedding(self.num_genes, hidden_size, max_norm=True) # 基因表达转为嵌入向量
        self.pert_emb = nn.Embedding(self.num_perts, hidden_size, max_norm=True) # 基因可能产生的扰动转为嵌入向量
        
        # 三个激活函数
        self.emb_trans = nn.ReLU()
        self.pert_base_trans = nn.ReLU()
        self.transform = nn.ReLU()
        self.emb_trans_v2 = MLP([hidden_size, hidden_size, hidden_size], last_layer_act='ReLU')
        # 共表达图后的基因表达
        self.pert_fuse = MLP([hidden_size, hidden_size, hidden_size], last_layer_act='ReLU')
        # 扰动求和后扰动向量表达
        
        # 传入计算好的共表达图
        self.G_coexpress = args['G_coexpress'].to(args['device'])
        # G_coexpress形状为(2, E), 记录E条边的起始点
        self.G_coexpress_weight = args['G_coexpress_weight'].to(args['device'])
        # G_coexpress_weight记录E条边的权重（Pearson相关系数）

        self.emb_pos = nn.Embedding(self.num_genes, hidden_size, max_norm=True)
        # 用于输入共表达图后的基因表达，转为嵌入向量
        self.layers_emb_pos = torch.nn.ModuleList()
        for i in range(1, self.num_layers_gene_pos + 1):  # 加入一个图卷积层
            self.layers_emb_pos.append(SGConv(hidden_size, hidden_size, 1))
            # SGConv图卷积层，该层进行1次图传播
        
        # GO图，同样传入边的起始点和权重
        self.G_sim = args['G_go'].to(args['device'])
        self.G_sim_weight = args['G_go_weight'].to(args['device'])

        self.sim_layers = torch.nn.ModuleList()
        for i in range(1, self.num_layers + 1):
            self.sim_layers.append(SGConv(hidden_size, hidden_size, 1))
            # SGConv图卷积层，该层进行1次图传播
        
        # 共享的MLP用于解码 64维 → 128维 → 64维
        self.recovery_w = MLP([hidden_size, hidden_size*2, hidden_size], last_layer_act='linear')
        
        # nn.Parameter 创建两套参数
        self.indv_w1 = nn.Parameter(torch.rand(self.num_genes,
                                               hidden_size, 1)) 
        # (5054, 64, 1) 5054个基因，64个特征，每个特征用1个值表示
        self.indv_b1 = nn.Parameter(torch.rand(self.num_genes, 1)) # 偏置
        # (5054, 1) 5054个基因位置，每个位置上有一个值表示扰动/变化
        self.act = nn.ReLU()
        nn.init.xavier_normal_(self.indv_w1)
        nn.init.xavier_normal_(self.indv_b1)
        # xavier_normal_：Xavier 正态初始化给参数赋值
        
        # 用先前的MLP类生成跨基因的MLP 5054维 → 64维 → 64维
        # 将初步预测分数再转入64维潜层
        self.cross_gene_state = MLP([self.num_genes, hidden_size,
                                     hidden_size])
        # 最终基因表达的解码参数
        self.indv_w2 = nn.Parameter(torch.rand(1, self.num_genes,
                                           hidden_size+1))
        # (1,5054,65) 初步分数一个维度加上跨基因的64个维度共65维
        self.indv_b2 = nn.Parameter(torch.rand(1, self.num_genes)) # 偏置
        nn.init.xavier_normal_(self.indv_w2)
        nn.init.xavier_normal_(self.indv_b2)
        
        # BN层
        self.bn_emb = nn.BatchNorm1d(hidden_size)
        self.bn_pert_base = nn.BatchNorm1d(hidden_size)
        self.bn_pert_base_trans = nn.BatchNorm1d(hidden_size)
        
        # 不确定性分支
        if self.uncertainty:
            self.uncertainty_w = MLP([hidden_size, hidden_size*2, hidden_size, 1], last_layer_act='linear')
        
    def forward(self, data):

        x, pert_idx = data.x, data.pert_idx
        # x为原表达，也是对照，pert_idx为每个细胞的各自扰动基因编号
        if self.no_perturb: # 不计算扰动变化
            out = x.reshape(-1,1)
            out = torch.split(torch.flatten(out), self.num_genes)           
            return torch.stack(out) # 相当于直接返回了(B, 5054)
        else:
            num_graphs = len(data.batch.unique()) # batch中可能出现同名，确定细胞数

            ## get base gene embeddings
            emb = self.gene_emb(torch.LongTensor(list(range(self.num_genes))).repeat(num_graphs, ).to(self.args['device']))  
            # 转嵌入向量，序列：根据基因数K，创建张量[1,2...K,1,2...K,1,2...K...重复batch次]      
            emb = self.bn_emb(emb) # BatchNorm
            base_emb = self.emb_trans(emb) # ReLU     

            pos_emb = self.emb_pos(torch.LongTensor(list(range(self.num_genes))).repeat(num_graphs, ).to(self.args['device']))
            # 共表达图后的嵌入向量
            for idx, layer in enumerate(self.layers_emb_pos):
                pos_emb = layer(pos_emb, self.G_coexpress, self.G_coexpress_weight)
                # 进行图卷积
                if idx < len(self.layers_emb_pos) - 1:
                    pos_emb = pos_emb.relu() # 如果没到最后一层，就执行 ReLU

            base_emb = base_emb + 0.2 * pos_emb
            base_emb = self.emb_trans_v2(base_emb)
            # 经过MLP融合基础表达和共表达后的表达

            pert_index = []
            for idx, i in enumerate(pert_idx): # 取出的i也是列表，对应各个细胞要扰动基因的序号
                for j in i:
                    if j != -1: # -1为无扰动
                        pert_index.append([idx, j])
                        # pert_index [[细胞1，基因1], [细胞2,基因1], [细胞2,基因2], [细胞3,基因1]]
            pert_index = torch.tensor(pert_index).T
            # 转置后：[[细胞1，细胞2，细胞2，细胞3],[基因1，基因1，基因2，基因1]]
            # pert_index[0] 细胞编号
            # pert_index[1] 扰动的基因编号

            pert_global_emb = self.pert_emb(torch.LongTensor(list(range(self.num_perts))).to(self.args['device']))        
            # 对扰动图中总计P种扰动基因构建embedding层

            for idx, layer in enumerate(self.sim_layers):
                pert_global_emb = layer(pert_global_emb, self.G_sim, self.G_sim_weight)
                # 进行图卷积
                if idx < self.num_layers - 1:
                    pert_global_emb = pert_global_emb.relu()


            base_emb = base_emb.reshape(num_graphs, self.num_genes, -1)
            # 先从(B*K, H)恢复(B, K, H)形状的基础基因表达

            if pert_index.shape[0] != 0:
                pert_track = {} # 用以保存细胞与之对应的扰动向量和
                for i, j in enumerate(pert_index[0]):
                    if j.item() in pert_track: # j.item()标量张量转换成Python数字
                        pert_track[j.item()] = pert_track[j.item()] + pert_global_emb[pert_index[1][i]]
                        # 相同细胞的多个扰动向量叠加作为value
                    else:
                        pert_track[j.item()] = pert_global_emb[pert_index[1][i]]
                        # key：j.item()  
                        # value：扰动的基因编号的列表中的第i个，也就是当前取出的j对应的扰动基因
                        # 送入pert_global_emb（pert_emb）

                if len(list(pert_track.values())) > 0: # 只要字典中有扰动向量
                    if len(list(pert_track.values())) == 1: # 只有一个有效扰动细胞
                        # 列表内容重复一次，主要是让 MLP 中的 BatchNorm 能接收到两个样本，避免后续报错
                        emb_total = self.pert_fuse(torch.stack(list(pert_track.values()) * 2))
                    else:
                        emb_total = self.pert_fuse(torch.stack(list(pert_track.values())))
                        # 取出每个细胞的（累加后的）扰动向量，转为列表再堆叠，送入pert_fuse（MLP）

                    for idx, j in enumerate(pert_track.keys()):
                        base_emb[j] = base_emb[j] + emb_total[idx]
                    # 细胞对应的基础表达加上对应的扰动向量

            base_emb = base_emb.reshape(num_graphs * self.num_genes, -1)
            # 恢复成(B*K, H)形状
            base_emb = self.bn_pert_base(base_emb) # BatchNorm

            base_emb = self.transform(base_emb) # ReLU       
            out = self.recovery_w(base_emb) # 共享的MLP 64 → 128 → 64
            out = out.reshape(num_graphs, self.num_genes, -1) # (B, K, H)
            out = out.unsqueeze(-1) * self.indv_w1 # 这里是数乘(B, K, H, 1) * (K, H ,1)本例是(5054, 64, 1)
            # 广播机制, 不足左侧补1 ,(B, K, H, 1) * (1, K, H ,1) 
            # out 变为(B, K, H, 1)
            w = torch.sum(out, axis = 2) # 沿H维求和(B, K, 1)
            out = w + self.indv_b1 # 加偏置

            # 跨基因
            cross_gene_embed = self.cross_gene_state(out.reshape(num_graphs, self.num_genes, -1).squeeze(2))
            # (B, K, 1) squeeze(2)删除第 2 维中长度为 1 的维度 -> (B, K)
            # cross_gene_state MLP (K → H → H)
            # 变为(B, H)
            cross_gene_embed = cross_gene_embed.repeat(1, self.num_genes)
            # (B, H) -> (B*1, H*K)

            cross_gene_embed = cross_gene_embed.reshape([num_graphs,self.num_genes, -1])
            # (B*1, H*K) -> (B, K, H)
            cross_gene_out = torch.cat([out, cross_gene_embed], 2)
            # (B, K, 1)拼接(B, K, H) -> (B, K, 1+H)
            # 1+H：自身分数加上整体的信息

            cross_gene_out = cross_gene_out * self.indv_w2
            # (B, K, 1+H) * (1, K, H+1) -> (B, K, H+1)
            cross_gene_out = torch.sum(cross_gene_out, axis=2)
            # (B, K, H+1) -> (B, K)
            out = cross_gene_out + self.indv_b2 # 加(1, K)偏置    
            out = out.reshape(num_graphs * self.num_genes, -1) + x.reshape(-1,1)
            # 转换为(B*K, 1)再加对照作为最终表达
            out = torch.split(torch.flatten(out), self.num_genes) # 展开转为元组
            # 每K个数分成一个细胞，元组里有B个长度为K的张量

            # 额外输出一个不确定性输出
            if self.uncertainty:
                out_logvar = self.uncertainty_w(base_emb) # (B*K, H)形状，只经过了BatchNorm和ReLU
                out_logvar = torch.split(torch.flatten(out_logvar), self.num_genes)
                return torch.stack(out), torch.stack(out_logvar)
            
            return torch.stack(out)